In [1]:

# ============================================================
# DATA REDUCTION - ALL MAJOR TECHNIQUES
# Banking Customer Dataset
# ============================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.cluster import KMeans

# ------------------------------------------------------------
# 1. CREATE BANKING DATASET
# ------------------------------------------------------------

data = {
    'Customer_ID': range(101, 121),
    'Age': [22, 35, 28, 45, 32, 40, 25, 50, 38, 29,
            42, 31, 55, 27, 48, 36, 33, 44, 26, 39],

    'Salary': [25000, 45000, 30000, 65000, 40000,
               55000, 28000, 80000, 50000, 32000,
               60000, 38000, 90000, 27000, 75000,
               48000, 42000, 68000, 29000, 52000],

    'Balance': [10000, 25000, 15000, 50000, 20000,
                35000, 12000, 70000, 30000, 18000,
                45000, 22000, 85000, 11000, 65000,
                28000, 24000, 55000, 13000, 40000],

    'Loan_Amount': [0, 50000, 20000, 100000, 30000,
                    60000, 0, 150000, 40000, 10000,
                    80000, 25000, 200000, 0, 120000,
                    45000, 35000, 90000, 0, 70000],

    'Transactions': [10, 25, 15, 40, 20, 30, 12, 50, 28, 18,
                     35, 22, 60, 11, 45, 27, 24, 42, 14, 32],

    'Gender': ['F', 'M', 'F', 'M', 'M', 'F', 'F', 'M', 'F', 'M',
               'F', 'M', 'F', 'F', 'M', 'F', 'M', 'M', 'F', 'M'],

    'City': ['Hyderabad', 'Chennai', 'Delhi', 'Mumbai', 'Hyderabad',
             'Chennai', 'Delhi', 'Mumbai', 'Hyderabad', 'Chennai',
             'Delhi', 'Mumbai', 'Hyderabad', 'Chennai', 'Delhi',
             'Mumbai', 'Hyderabad', 'Chennai', 'Delhi', 'Mumbai']
}

df = pd.DataFrame(data)

print("ORIGINAL DATASET")
display(df)

print("Original Shape:", df.shape)


# ------------------------------------------------------------
# 2. FEATURE SELECTION
# ------------------------------------------------------------

# Select only required columns for analysis.
# Customer_ID and City are excluded in this example.

selected_features = df[
    ['Age', 'Salary', 'Balance', 'Loan_Amount', 'Transactions']
]

print("\n2. FEATURE SELECTION")
display(selected_features)


# ------------------------------------------------------------
# 3. RANDOM SAMPLING
# ------------------------------------------------------------

# Randomly selects 50% of the records.
# This reduces the number of rows.

random_sample = df.sample(frac=0.5, random_state=42)

print("\n3. RANDOM SAMPLING")
display(random_sample)

print("Original Rows:", len(df))
print("Sampled Rows:", len(random_sample))


# ------------------------------------------------------------
# 4. STRATIFIED SAMPLING
# ------------------------------------------------------------

# Selects samples while maintaining the same Gender proportion.
# Useful when different categories must be represented fairly.

stratified_sample, _ = train_test_split(
    df,
    train_size=0.5,
    stratify=df['Gender'],
    random_state=42
)

print("\n4. STRATIFIED SAMPLING")
display(stratified_sample)

print("Gender Distribution in Original Data:")
print(df['Gender'].value_counts(normalize=True))

print("Gender Distribution in Sample:")
print(stratified_sample['Gender'].value_counts(normalize=True))


# ------------------------------------------------------------
# 5. SYSTEMATIC SAMPLING
# ------------------------------------------------------------

# Selects every second record from the dataset.

systematic_sample = df.iloc[::2]

print("\n5. SYSTEMATIC SAMPLING")
display(systematic_sample)


# ------------------------------------------------------------
# 6. VARIANCE THRESHOLD
# ------------------------------------------------------------

# Removes features having zero variance.
# A zero-variance feature contains the same value in every row.

numeric_data = df[
    ['Age', 'Salary', 'Balance', 'Loan_Amount', 'Transactions']
]

selector = VarianceThreshold(threshold=0.0)

variance_result = selector.fit_transform(numeric_data)

selected_columns = selector.get_feature_names_out(
    numeric_data.columns
)

print("\n6. VARIANCE THRESHOLD")
print("Selected Features:", selected_columns)

display(pd.DataFrame(
    variance_result,
    columns=selected_columns
))


# ------------------------------------------------------------
# 7. CORRELATION-BASED FEATURE SELECTION
# ------------------------------------------------------------

# Finds relationships between numerical features.
# Highly correlated features may contain overlapping information.

correlation = numeric_data.corr()

print("\n7. CORRELATION MATRIX")
display(correlation)

# Identify pairs with absolute correlation above 0.90.

upper_triangle = correlation.abs().where(
    np.triu(np.ones(correlation.shape), k=1).astype(bool)
)

high_correlation = [
    column
    for column in upper_triangle.columns
    if any(upper_triangle[column] > 0.90)
]

print("Highly Correlated Features:", high_correlation)


# ------------------------------------------------------------
# 8. PRINCIPAL COMPONENT ANALYSIS (PCA)
# ------------------------------------------------------------

# PCA reduces multiple numerical features into fewer components.
# First, standardize the data.

scaler = StandardScaler()

scaled_data = scaler.fit_transform(numeric_data)

# Reduce five features into two principal components.

pca = PCA(n_components=2)

pca_result = pca.fit_transform(scaled_data)

pca_df = pd.DataFrame(
    pca_result,
    columns=['PC1', 'PC2']
)

print("\n8. PRINCIPAL COMPONENT ANALYSIS")
display(pca_df)

print("Explained Variance Ratio:")
print(pca.explained_variance_ratio_)

print("Total Information Retained:",
      pca.explained_variance_ratio_.sum())


# ------------------------------------------------------------
# 9. SINGULAR VALUE DECOMPOSITION (SVD)
# ------------------------------------------------------------

# SVD decomposes a matrix into smaller mathematical components.
# It can reduce the number of dimensions.

svd = TruncatedSVD(n_components=2, random_state=42)

svd_result = svd.fit_transform(scaled_data)

svd_df = pd.DataFrame(
    svd_result,
    columns=['SVD1', 'SVD2']
)

print("\n9. SINGULAR VALUE DECOMPOSITION")
display(svd_df)

print("Explained Variance Ratio:")
print(svd.explained_variance_ratio_)


# ------------------------------------------------------------
# 10. DATA AGGREGATION
# ------------------------------------------------------------

# Groups records by City and calculates average salary and balance.
# Multiple customer records are summarized into fewer records.

aggregated_data = df.groupby('City').agg({
    'Salary': 'mean',
    'Balance': 'mean',
    'Loan_Amount': 'sum',
    'Transactions': 'sum'
}).reset_index()

print("\n10. DATA AGGREGATION")
display(aggregated_data)

print("Original Rows:", len(df))
print("Aggregated Rows:", len(aggregated_data))


# ------------------------------------------------------------
# 11. NUMEROSITY REDUCTION USING K-MEANS
# ------------------------------------------------------------

# K-Means groups similar customer records into clusters.
# Cluster centers summarize groups of customers.

kmeans = KMeans(
    n_clusters=3,
    random_state=42,
    n_init=10
)

df_cluster = numeric_data.copy()

df_cluster['Cluster'] = kmeans.fit_predict(
    scaled_data
)

print("\n11. K-MEANS CLUSTERING")
display(df_cluster)

# Display cluster centers in original feature scale.

cluster_centers = scaler.inverse_transform(
    kmeans.cluster_centers_
)

centers_df = pd.DataFrame(
    cluster_centers,
    columns=numeric_data.columns
)

print("Cluster Centers:")
display(centers_df)


# ------------------------------------------------------------
# 12. DATA COMPRESSION
# ------------------------------------------------------------

# Converts a DataFrame into a compressed ZIP file.
# This reduces storage space, not the number of features.

df.to_csv('banking_data.csv', index=False)

df.to_csv(
    'banking_data_compressed.csv.gz',
    index=False,
    compression='gzip'
)

import os

original_size = os.path.getsize('banking_data.csv')
compressed_size = os.path.getsize('banking_data_compressed.csv.gz')

print("\n12. DATA COMPRESSION")
print("Original File Size:", original_size, "bytes")
print("Compressed File Size:", compressed_size, "bytes")


# ------------------------------------------------------------
# 13. FINAL COMPARISON
# ------------------------------------------------------------

print("\n13. FINAL DATA REDUCTION SUMMARY")

print("Original Dataset Shape:", df.shape)
print("Feature Selection Shape:", selected_features.shape)
print("Random Sampling Shape:", random_sample.shape)
print("Stratified Sampling Shape:", stratified_sample.shape)
print("Systematic Sampling Shape:", systematic_sample.shape)
print("PCA Reduced Shape:", pca_df.shape)
print("SVD Reduced Shape:", svd_df.shape)
print("Aggregated Dataset Shape:", aggregated_data.shape)

print("\nALL DATA REDUCTION TECHNIQUES COMPLETED!")


ORIGINAL DATASET


,Customer_ID,Age,Salary,Balance,Loan_Amount,Transactions,Gender,City
0,101,22,25000,10000,0,10,F,Hyderabad
1,102,35,45000,25000,50000,25,M,Chennai
2,103,28,30000,15000,20000,15,F,Delhi
3,104,45,65000,50000,100000,40,M,Mumbai
4,105,32,40000,20000,30000,20,M,Hyderabad
5,106,40,55000,35000,60000,30,F,Chennai
6,107,25,28000,12000,0,12,F,Delhi
7,108,50,80000,70000,150000,50,M,Mumbai
8,109,38,50000,30000,40000,28,F,Hyderabad
9,110,29,32000,18000,10000,18,M,Chennai


Original Shape: (20, 8)

2. FEATURE SELECTION


,Age,Salary,Balance,Loan_Amount,Transactions
0,22,25000,10000,0,10
1,35,45000,25000,50000,25
2,28,30000,15000,20000,15
3,45,65000,50000,100000,40
4,32,40000,20000,30000,20
5,40,55000,35000,60000,30
6,25,28000,12000,0,12
7,50,80000,70000,150000,50
8,38,50000,30000,40000,28
9,29,32000,18000,10000,18



3. RANDOM SAMPLING


,Customer_ID,Age,Salary,Balance,Loan_Amount,Transactions,Gender,City
0,101,22,25000,10000,0,10,F,Hyderabad
17,118,44,68000,55000,90000,42,M,Chennai
15,116,36,48000,28000,45000,27,F,Mumbai
1,102,35,45000,25000,50000,25,M,Chennai
8,109,38,50000,30000,40000,28,F,Hyderabad
5,106,40,55000,35000,60000,30,F,Chennai
11,112,31,38000,22000,25000,22,M,Mumbai
3,104,45,65000,50000,100000,40,M,Mumbai
18,119,26,29000,13000,0,14,F,Delhi
16,117,33,42000,24000,35000,24,M,Hyderabad


... (15 outputs left out)

,Age,Salary,Balance,Loan_Amount,Transactions,Cluster
0,22,25000,10000,0,10,1
1,35,45000,25000,50000,25,1
2,28,30000,15000,20000,15,1
3,45,65000,50000,100000,40,2
4,32,40000,20000,30000,20,1
5,40,55000,35000,60000,30,2
6,25,28000,12000,0,12,1
7,50,80000,70000,150000,50,0
8,38,50000,30000,40000,28,2
9,29,32000,18000,10000,18,1


Cluster Centers:


,Age,Salary,Balance,Loan_Amount,Transactions
0,51.000000,81666.666667,73333.333333,156666.666667,51.666667
1,28.800000,33600.000000,17000.000000,17000.000000,17.100000
2,40.571429,56857.142857,40428.571429,69285.714286,33.428571



12. DATA COMPRESSION
Original File Size: 811 bytes
Compressed File Size: 401 bytes

13. FINAL DATA REDUCTION SUMMARY
Original Dataset Shape: (20, 8)
Feature Selection Shape: (20, 5)
Random Sampling Shape: (10, 8)
Stratified Sampling Shape: (10, 8)
Systematic Sampling Shape: (10, 8)
PCA Reduced Shape: (20, 2)
SVD Reduced Shape: (20, 2)
Aggregated Dataset Shape: (4, 5)

ALL DATA REDUCTION TECHNIQUES COMPLETED!
